# HyperOpt

https://hyperopt.github.io/hyperopt/

HyperOpt는 하이퍼파라미터 탐색을 더 효율적으로 수행하기 위한 최적화 라이브러리이다.

앞에서 Grid Search, Random Search처럼 여러 하이퍼파라미터 조합을 시도해 보는 방법을 보았다.
이제 HyperOpt에서는 모든 조합을 단순히 다 해보는 것이 아니라, 이전 시도의 결과를 바탕으로 다음 탐색 위치를 더 똑똑하게 정하는 방식에 주목한다.

1. 하이퍼파라미터 튜닝은 성능 향상에 중요하지만, 모든 조합을 전부 실험하는 것은 시간과 비용이 많이 든다.
2. HyperOpt는 이전 실험 결과를 바탕으로 다음 후보를 더 효율적으로 탐색하는 베이지안 최적화 계열 라이브러리이다.
3. 따라서 단순히 best parameter만 보는 것이 아니라, 탐색 공간을 어떻게 정의하는지와 반복 탐색이 어떤 방식으로 이루어지는지도 함께 이해하는 것이 중요하다.

## 왜 HyperOpt를 사용하는가?

Grid Search는 정해둔 후보를 전부 다 확인한다.
그래서 탐색 범위가 넓어질수록 경우의 수가 급격히 늘어난다.

Random Search는 후보를 무작위로 뽑아서 시도한다.
Grid Search보다 덜 비효율적일 수 있지만, 이전 결과를 바탕으로 다음 탐색을 조정하지는 않는다.

HyperOpt는 이전 실험 결과를 참고하여, 성능이 좋을 가능성이 높은 방향을 다음 탐색에 반영한다.
즉, 적은 시도로도 괜찮은 하이퍼파라미터 조합을 찾도록 돕는 것이 핵심이다.

## HyperOpt의 핵심 아이디어

1. 탐색할 하이퍼파라미터 범위를 먼저 정의한다.
2. 몇 번의 실험을 통해 각 조합의 성능을 확인한다.
3. 이전 실험 결과를 바탕으로, 다음에는 어디를 시도할지 더 유망한 후보를 선택한다.
4. 이 과정을 반복하면서 더 좋은 조합을 찾아간다.

즉, "지금까지의 결과를 보고 다음 시도를 조정한다"는 점이 HyperOpt의 핵심이다.

## HyperOpt에서 함께 봐야 할 것

1. search space
   * 어떤 하이퍼파라미터를 어떤 범위에서 탐색할지 정의한다.
   * 예를 들어 learning_rate는 0.01 ~ 0.3 사이, max_depth는 3 ~ 10 사이처럼 설정할 수 있다.

2. objective function
   * 주어진 하이퍼파라미터 조합이 얼마나 좋은지 점수로 반환하는 함수이다.
   * 보통 검증 데이터의 accuracy, f1-score, RMSE 같은 평가 지표를 사용한다.

3. trials
   * 각 탐색 결과를 기록하는 객체이다.
   * 어떤 조합을 시도했고, 그 결과가 어땠는지 추적할 수 있다.

4. fmin
   * HyperOpt에서 최적화를 실제로 수행하는 함수이다.
   * 정의한 탐색 공간과 objective function을 바탕으로 최적의 조합을 찾는다.

## HyperOpt의 장점

1. 효율적인 탐색
   * 모든 조합을 전부 시도하지 않아도 된다.
   * 이전 결과를 반영하여 더 가능성 있는 후보를 우선적으로 탐색한다.

2. 복잡한 탐색 공간 처리
   * 정수형, 실수형, 범주형 하이퍼파라미터를 함께 정의할 수 있다.
   * 조건부 탐색 공간도 비교적 유연하게 구성할 수 있다.

3. 실무 적용에 적합
   * 모델 학습 비용이 큰 경우, 제한된 횟수 안에서 합리적인 조합을 찾는 데 유리하다.
   * 특히 부스팅 계열 모델처럼 튜닝할 파라미터가 많은 경우 유용하다.

## 주의할 점

1. HyperOpt가 항상 최적해를 보장하는 것은 아니다.
   * 제한된 횟수 안에서 더 좋은 후보를 효율적으로 찾도록 돕는 방식에 가깝다.

2. 탐색 공간을 너무 넓게 잡으면 비효율적일 수 있다.
   * 어떤 파라미터를 어느 범위에서 탐색할지 적절히 설계하는 것이 중요하다.

3. objective function을 어떻게 정의하느냐가 중요하다.
   * 검증 방식이나 평가 지표가 부적절하면, 찾은 하이퍼파라미터도 좋은 결과로 이어지지 않을 수 있다.

## 환경설정
HyperOpt는 내부적으로 pkg_resources에 의존하는 버전 이슈가 있어, 일부 환경에서는 setuptools 버전을 낮춰야 정상 동작할 수 있다.

In [ ]:
# %conda install hyperopt
# %pip install "setuptools<82"

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### iris + RandomForestClassfier

In [2]:
# 데이터 준비
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(120, 4) (30, 4)
target
0    40
2    40
1    40
Name: count, dtype: int64


In [ ]:
# HyperOpt 탐색 공간 정의
from hyperopt import hp

n_estimators_vals = [100, 300, 500]
max_depth_vals = [None, 5, 10]
min_samples_split_vals = [2, 5, 10]
min_samples_leaf_vals = [1, 2, 4]

# hp.choice(label, options) : options에 들어있는 후보 중 하나를 고른다
search_space = {
    'n_estimators' : hp.choice('n_estimators', n_estimators_vals),
    'max_depth' : hp.choice('max_depth', max_depth_vals),
    'min_samples_split' : hp.choice('min_samples_split', min_samples_split_vals),
    'min_samples_leaf' : hp.choice('min_samples_leaf', min_samples_leaf_vals),
}

search_space

{'n_estimators': <hyperopt.pyll.base.Apply at 0x26288d62480>,
 'max_depth': <hyperopt.pyll.base.Apply at 0x26288d63da0>,
 'min_samples_split': <hyperopt.pyll.base.Apply at 0x26288d609b0>,
 'min_samples_leaf': <hyperopt.pyll.base.Apply at 0x26288d60350>}

In [ ]:
from hyperopt.pyll.stochastic import sample

for i in range(5):
    sampled_values = sample(search_space)   # 실제로 어떤 후보가 생성 될 것인지 미리 확인하는 용도 - 탐색 공간 검수
    print(f'[[{i+1}]]: {sampled_values}')

[[1]]: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
[[2]]: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}
[[3]]: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
[[4]]: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
[[5]]: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}


In [6]:
# objective function 정의
# HyperOpt의 fmin()은 "loss 최소화" 방향으로 동작하며 원하는 기준을 함수에 정의해서 설정한다
from hyperopt import STATUS_OK
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def objective(params):

    # search space에서 뽑힌 하이퍼파라미터 조합으로 설정 된 모델
    model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)

    # train 데이터 내부의 교차 검증 점수 계산
    cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)

    # 평균 값
    mean_acc = cv_scores.mean()

    return {
        'loss' : -mean_acc,     # fmin()은 loss를 최소화하므로 acc(정확도)를 음수로 변환해서 반환
        'status' : STATUS_OK    # 해당 실험이 정상적으로 끝났음을 의미하는 상태값
    }


In [7]:
# HyperOpt 실행
from hyperopt import Trials, fmin, tpe

trials = Trials()

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    trials=trials,
    max_evals=20,
    rstate=np.random.default_rng(seed=42)
)

print('best_result (choice 인덱스 기준):', best_result)
print('평가 횟수:', len(trials.trials))

100%|██████████| 20/20 [00:31<00:00,  1.58s/trial, best loss: -0.9666666666666667]
best_result (choice 인덱스 기준): {'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 0}
평가 횟수: 20


In [8]:
# trials 결과 확인
trials_df = pd.DataFrame({
    'n_estimators_idx': trials.vals['n_estimators'],
    'max_depth_idx': trials.vals['max_depth'],
    'min_samples_split_idx': trials.vals['min_samples_split'],
    'min_samples_leaf_idx': trials.vals['min_samples_leaf'],
    'loss': [result['loss'] for result in trials.results]
})

trials_df['n_estimators'] = trials_df['n_estimators_idx'].apply(lambda x: n_estimators_vals[x])
trials_df['max_depth'] = trials_df['max_depth_idx'].apply(lambda x: max_depth_vals[x])
trials_df['min_samples_split'] = trials_df['min_samples_split_idx'].apply(lambda x: min_samples_split_vals[x])
trials_df['min_samples_leaf'] = trials_df['min_samples_leaf_idx'].apply(lambda x: min_samples_leaf_vals[x])

# loss는 -accuracy 이므로, 보기 쉽게 accuracy로 다시 바꾼 열 추가
trials_df['cv_accuracy'] = -trials_df['loss']

trials_df = trials_df[
    ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'cv_accuracy', 'loss']
].sort_values(by='cv_accuracy', ascending=False)

trials_df

,n_estimators,max_depth,min_samples_split,min_samples_leaf,cv_accuracy,loss
0,100,5.0,10,2,0.966667,-0.966667
12,300,10.0,2,2,0.966667,-0.966667
17,300,NaN,2,4,0.966667,-0.966667
16,500,10.0,10,4,0.966667,-0.966667
15,100,10.0,10,2,0.966667,-0.966667
5,500,NaN,10,2,0.966667,-0.966667
13,500,5.0,2,2,0.966667,-0.966667
7,300,10.0,2,4,0.966667,-0.966667
8,100,10.0,10,2,0.966667,-0.966667
9,500,5.0,2,2,0.966667,-0.966667


In [9]:
# best_result를 실제 파라미터 값으로 복원
best_params = {
    'n_estimators': n_estimators_vals[best_result['n_estimators']],
    'max_depth': max_depth_vals[best_result['max_depth']],
    'min_samples_split': min_samples_split_vals[best_result['min_samples_split']],
    'min_samples_leaf': min_samples_leaf_vals[best_result['min_samples_leaf']],
}

print(best_params)

{'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 2}
